In [1]:
# Cell 1: Read the bronze_sales Delta table
df = spark.read.format('delta').load('Tables/dbo/bronze_sales')

# Show the first 10 rows
df.show(10)



StatementMeta(, e61d8c97-2ded-4a8d-9ee0-538fc713b04c, 3, Finished, Available, Finished, False)

+---------+----------+----------------+------+---------------+--------+--------+---------+
|  OrderID| OrderDate|    CustomerName|Region|ProductCategory| Revenue|Quantity|   Status|
+---------+----------+----------------+------+---------------+--------+--------+---------+
|ORD-01255|18-08-2024|   Gaurav Mishra| South|      Furniture|     0.0|       3| Returned|
|ORD-01314|23-10-2023|     Aakash Bhat| South|      Furniture| 5911.62|       6|   Active|
|ORD-01042|29-08-2023|Rajesh Choudhury| South|Office Supplies|  899.44|      20|   Active|
|ORD-01709|08-01-2023|    Ajay Rastogi| North|Office Supplies|   74.82|      21|   Active|
|ORD-01484|04-11-2024|      Divya Kaur| North|Office Supplies| 1110.91|       7|   Active|
|ORD-01443|10-04-2024|    Geeta Pandey|  West|    Electronics|     0.0|       1|Cancelled|
|ORD-02004|22-08-2023|     Sneha Gupta|  East|      Furniture|15696.99|       7|   Active|
|ORD-01927|19-12-2023|     Arjun Reddy| South|Office Supplies| 3453.15|      13|   Active|

In [2]:
#Cell 2: Check Schema
df.printSchema()

StatementMeta(, e61d8c97-2ded-4a8d-9ee0-538fc713b04c, 4, Finished, Available, Finished, False)

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- ProductCategory: string (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Status: string (nullable = true)



In [3]:
# Cell 3: Transformation 1 - Filter rows
# Keep only active orders with positive revenue
df_filtered = df.filter(
    (df['Status'] == 'Active') & (df['Revenue'] > 0)
)

# Check how many rows remain
print(f'Original row count: {df.count()}')
print(f'Filtered row count: {df_filtered.count()}')


StatementMeta(, e61d8c97-2ded-4a8d-9ee0-538fc713b04c, 5, Finished, Available, Finished, False)

Original row count: 1525
Filtered row count: 1268


In [4]:
# Cell 4: Transformation 2 - Rename columns to snake_case
df_renamed = df_filtered \
    .withColumnRenamed('OrderID', 'order_id') \
    .withColumnRenamed('OrderDate', 'order_date') \
    .withColumnRenamed('CustomerName', 'customer_name') \
    .withColumnRenamed('Region', 'region') \
    .withColumnRenamed('ProductCategory', 'product_category') \
    .withColumnRenamed('Revenue', 'revenue') \
    .withColumnRenamed('Quantity', 'quantity') \
    .withColumnRenamed('Status', 'status')

display(df_renamed.limit(5))


StatementMeta(, e61d8c97-2ded-4a8d-9ee0-538fc713b04c, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ed1748dd-cdcf-4ff2-bb31-7607d082850c)

In [5]:
# Fix order_date from string to proper DateType
from pyspark.sql.functions import to_date, col

df_renamed = df_renamed.withColumn(
    'order_date',
    to_date(col('order_date'), 'dd-MM-yyyy')
)

# Verify it worked
print(df_renamed.schema['order_date'].dataType)
display(df_renamed.select('order_id', 'order_date').limit(5))


StatementMeta(, e61d8c97-2ded-4a8d-9ee0-538fc713b04c, 7, Finished, Available, Finished, False)

DateType()


SynapseWidget(Synapse.DataFrame, 83898be7-343b-4b93-8379-ffd465f1dbba)

In [6]:
# Cell 5: Transformation 3 - Add calculated column & cast date type
from pyspark.sql.functions import col, round, to_date

# Apply a fixed INR to USD conversion rate (example: 1 USD = 83 INR)
EXCHANGE_RATE = 83.0

df_transformed = df_renamed \
    .withColumn('revenue_usd', round(col('revenue') / EXCHANGE_RATE, 2)) \

# Verify the data types and values
df_transformed.printSchema()
display(df_transformed.select('order_id', 'order_date', 'revenue', 'revenue_usd').limit(10))


StatementMeta(, e61d8c97-2ded-4a8d-9ee0-538fc713b04c, 8, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- revenue_usd: double (nullable = true)



SynapseWidget(Synapse.DataFrame, 849748c1-c93b-47b4-b2aa-6ce5e677f8b6)

In [7]:
# Cell 6: Write the transformed DataFrame as silver_sales Delta table
df_transformed.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('silver_sales')


StatementMeta(, e61d8c97-2ded-4a8d-9ee0-538fc713b04c, 9, Finished, Available, Finished, False)